In [4]:
#task 1

total_cards = 52

red_cards = 26

hearts = 13

face_cards = 12

diamond_face = 3

spade_face = 3

queens = 4

overlap = 1

p_red = red_cards / total_cards
print("P(Red Card):", p_red)

p_heart_given_red = hearts / red_cards
print("P(Heart | Red):", p_heart_given_red)

p_diamond_given_face = diamond_face / face_cards
print("P(Diamond | Face Card):", p_diamond_given_face)

p_spade_or_queen = (spade_face + queens - overlap) / face_cards
print("P(Spade OR Queen | Face Card):", p_spade_or_queen)

P(Red Card): 0.5
P(Heart | Red): 0.5
P(Diamond | Face Card): 0.25
P(Spade OR Queen | Face Card): 0.5


In [9]:
!pip install pgmpy
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# Step 1: Define structure
model = DiscreteBayesianNetwork([
    ('I', 'G'),
    ('S', 'G'),
    ('D', 'G'),
    ('G', 'P')
])

# Step 2: Define CPDs

cpd_I = TabularCPD('I', 2, [[0.7], [0.3]])
cpd_S = TabularCPD('S', 2, [[0.6], [0.4]])
cpd_D = TabularCPD('D', 2, [[0.6], [0.4]])

cpd_G = TabularCPD(
    variable='G', variable_card=3,
    values=[
        [0.7, 0.5, 0.4, 0.2, 0.3, 0.2, 0.1, 0.05],
        [0.2, 0.3, 0.4, 0.5, 0.4, 0.3, 0.3, 0.25],
        [0.1, 0.2, 0.2, 0.3, 0.3, 0.5, 0.6, 0.7]
    ],
    evidence=['I', 'S', 'D'],
    evidence_card=[2, 2, 2]
)

cpd_P = TabularCPD(
    variable='P', variable_card=2,
    values=[
        [0.95, 0.80, 0.50],
        [0.05, 0.20, 0.50]
    ],
    evidence=['G'],
    evidence_card=[3]
)

model.add_cpds(cpd_I, cpd_S, cpd_D, cpd_G, cpd_P)

print("Model valid:", model.check_model())

inference = VariableElimination(model)

q1 = inference.query(variables=['P'], evidence={'S': 0, 'D': 1})
print("\nP(Pass | Sufficient, Hard):")
print(q1)

q2 = inference.query(variables=['I'], evidence={'P': 0})
print("\nP(Intelligence | Pass=Yes):")
print(q2)

Model valid: True

P(Pass | Sufficient, Hard):
+------+----------+
| P    |   phi(P) |
+======+==========+
| P(0) |   0.7745 |
+------+----------+
| P(1) |   0.2255 |
+------+----------+

P(Intelligence | Pass=Yes):
+------+----------+
| I    |   phi(I) |
+======+==========+
| I(0) |   0.7372 |
+------+----------+
| I(1) |   0.2628 |
+------+----------+


In [12]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('Disease', 'Fever'),
    ('Disease', 'Cough'),
    ('Disease', 'Fatigue'),
    ('Disease', 'Chills')
])

cpd_D = TabularCPD(variable='Disease', variable_card=2,
                   values=[[0.3], [0.7]])

# Fever
cpd_Fever = TabularCPD(
    variable='Fever', variable_card=2,
    values=[[0.9, 0.5], [0.1, 0.5]],
    evidence=['Disease'],
    evidence_card=[2]
)

cpd_Cough = TabularCPD(
    variable='Cough', variable_card=2,
    values=[[0.8, 0.6], [0.2, 0.4]],
    evidence=['Disease'],
    evidence_card=[2]
)

cpd_Fatigue = TabularCPD(
    variable='Fatigue', variable_card=2,
    values=[[0.7, 0.3], [0.3, 0.7]],
    evidence=['Disease'],
    evidence_card=[2]
)

cpd_Chills = TabularCPD(
    variable='Chills', variable_card=2,
    values=[[0.6, 0.4], [0.4, 0.6]],
    evidence=['Disease'],
    evidence_card=[2]
)

model.add_cpds(cpd_D, cpd_Fever, cpd_Cough, cpd_Fatigue, cpd_Chills)

print("Model valid:", model.check_model())

inference = VariableElimination(model)

q1 = inference.query(variables=['Disease'],
                     evidence={'Fever': 0, 'Cough': 0})
print("\nP(Disease | Fever, Cough):")
print(q1)

q2 = inference.query(variables=['Disease'],
                     evidence={'Fever': 0, 'Cough': 0, 'Chills': 0})
print("\nP(Disease | Fever, Cough, Chills):")
print(q2)

q3 = inference.query(variables=['Fatigue'],
                     evidence={'Disease': 0})
print("\nP(Fatigue | Flu):")


Model valid: True

P(Disease | Fever, Cough):
+------------+----------------+
| Disease    |   phi(Disease) |
+============+================+
| Disease(0) |         0.5070 |
+------------+----------------+
| Disease(1) |         0.4930 |
+------------+----------------+

P(Disease | Fever, Cough, Chills):
+------------+----------------+
| Disease    |   phi(Disease) |
+============+================+
| Disease(0) |         0.6067 |
+------------+----------------+
| Disease(1) |         0.3933 |
+------------+----------------+

P(Fatigue | Flu):


In [6]:
#task 4

import numpy as np

states = ["Sunny", "Cloudy", "Rainy"]

transition = np.array([
    [0.6, 0.3, 0.1],  # Sunny
    [0.3, 0.4, 0.3],  # Cloudy
    [0.2, 0.3, 0.5]   # Rainy
])

current = 0
days = []

for i in range(10):
    current = np.random.choice([0,1,2], p=transition[current])
    days.append(states[current])

print("Weather for 10 days:")
print(days)

rainy_days = days.count("Rainy")
print("Rainy days:", rainy_days)

count = 0
trials = 10000

for _ in range(trials):
    current = 0
    rainy = 0
    for i in range(10):
        current = np.random.choice([0,1,2], p=transition[current])
        if states[current] == "Rainy":
            rainy += 1
    if rainy >= 3:
        count += 1

print("P(at least 3 rainy days):", count / trials)

Weather for 10 days:
['Cloudy', 'Rainy', 'Rainy', 'Rainy', 'Rainy', 'Rainy', 'Rainy', 'Sunny', 'Sunny', 'Sunny']
Rainy days: 6
P(at least 3 rainy days): 0.4405
